# Image Clustering with Vision Transformers
Extract pretrained ViT embeddings, normalize them, and group images with K-Means or Agglomerative Clustering. Add your images to `images/` before running.

In [ ]:
%pip install -q transformers scikit-learn pillow

In [ ]:
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from transformers import AutoImageProcessor, ViTModel
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score


In [ ]:
IMAGE_DIR = Path('images')
MODEL_NAME = 'google/vit-base-patch16-224'
N_CLUSTERS = 5
METHOD = 'kmeans'  # 'kmeans' or 'agglomerative'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = ViTModel.from_pretrained(MODEL_NAME).to(device).eval()


In [ ]:
extensions = {'.jpg', '.jpeg', '.png', '.webp'}
image_paths = sorted(p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in extensions)
features, valid_paths = [], []
with torch.inference_mode():
    for path in image_paths:
        image = Image.open(path).convert('RGB')
        inputs = processor(images=image, return_tensors='pt').to(device)
        vector = model(**inputs).last_hidden_state[:, 0, :].squeeze(0)
        features.append(vector.cpu().numpy())
        valid_paths.append(path)
embeddings = normalize(np.asarray(features))
print(f'Extracted {len(valid_paths)} embeddings with shape {embeddings.shape}')


In [ ]:
if not 2 <= N_CLUSTERS <= len(valid_paths):
    raise ValueError('N_CLUSTERS must be between 2 and the number of images')
if METHOD == 'kmeans':
    clusterer = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init='auto')
else:
    clusterer = AgglomerativeClustering(n_clusters=N_CLUSTERS, metric='cosine', linkage='average')
labels = clusterer.fit_predict(embeddings)
print('Silhouette score:', silhouette_score(embeddings, labels, metric='cosine') if len(set(labels)) > 1 and len(set(labels)) < len(labels) else 'Not defined')


In [ ]:
import pandas as pd
results = pd.DataFrame({'image': [p.name for p in valid_paths], 'cluster': labels})
display(results.sort_values(['cluster', 'image']))
results.to_csv('cluster_assignments.csv', index=False)


## Interpretation
Cluster numbers are arbitrary. Inspect the grouped images visually before assigning semantic labels. The silhouette score measures geometric separation in embedding space, not whether the groups are meaningful to a person.